## 量化FP32模型并导出单个测试图片计算过程中的所有INT8数据
1. **其中Scale参数来源于训练集采样** 

In [2]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

# ==========================================
# 0. 定义手动展平函数
# ==========================================
def my_flatten(x):
    """
    手动实现展平操作，严格按照 C/C++ 的行优先 (Row-Major) 逻辑。
    """
    B, C, H, W = x.shape
    flattened_memory = []
    for b in range(B):
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    val = x[b, c, h, w].item() 
                    flattened_memory.append(val)
    return torch.tensor([flattened_memory], dtype=x.dtype, device=x.device)

# ==========================================
# 1. 融合 BN 后的纯净全网定义 (Hardware-Ready)
# ==========================================
class HardwareFriendlyCNNFused(torch.nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNNFused, self).__init__()
        # 直接读取融合后的通道数配置: 3->64->64->64->64
        self.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = torch.nn.MaxPool2d(2, 2)
        
        self.conv3 = torch.nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv4 = torch.nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool2 = torch.nn.MaxPool2d(2, 2)
        
        self.fc = torch.nn.Linear(64 * 8 * 8, 10)
        self.relu = torch.nn.ReLU()
        # 推理时不需要 dropout，这里删去
        
    def forward(self, x):
        # 严格对齐 train_cnn.py 里的前向计算逻辑
        x = self.relu(self.conv1(x))
        x = self.pool1(self.relu(self.conv2(x)))
        x = self.relu(self.conv3(x))
        x = self.pool2(self.relu(self.conv4(x)))
        x = x.view(x.size(0), -1) 
        x = self.fc(x)
        return x

# ==========================================
# 2. 量化核心工具函数
# ==========================================
def sym_quantize(tensor):
    """对称量化：强制 Zero Point = 0"""
    max_abs = tensor.abs().max().item()
    scale = max_abs / 127.0 if max_abs > 0 else 1.0
    q_tensor = torch.round(tensor / scale)
    q_tensor = torch.clamp(q_tensor, -127, 127).to(torch.int8)
    return q_tensor, scale

def export_c_array(tensor, name, file_handle, c_type="int8_t"):
    """导出为一维 C 数组"""
    flat_data = tensor.flatten().tolist()
    file_handle.write(f"const {c_type} {name}[{len(flat_data)}] = {{\n    ")
    for i, val in enumerate(flat_data):
        file_handle.write(f"{val}, ")
        if (i + 1) % 16 == 0:
            file_handle.write("\n    ")
    file_handle.write("\n};\n\n")

def calc_multiplier_shift(s_in, s_w, s_out, shift=16):
    """计算硬件所需的 Multiplier，Shift 固定为 16"""
    M = (s_in * s_w) / s_out
    multiplier = int(round(M * (1 << shift)))
    return multiplier, shift

# ==========================================
# 3. 主流程：计算 Scale，生成黄金模型
# ==========================================
def main():
    print(">>> 步骤 1/4: 加载融合版模型与数据集...")
    model = HardwareFriendlyCNNFused()
    # 读取上一步生成的无 BN 纯权重！
    model.load_state_dict(torch.load('my_hardware_cnn_fp32_64ch_fused.pth', weights_only=True))
    model.eval()
    
    transform = transforms.Compose([transforms.ToTensor()])
    
    # 训练集用于校准 PTQ 的极值
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    calib_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(trainset, range(1000)), batch_size=100, shuffle=False)
    
    # 测试集仅用于提取 Index=1 作打靶测试
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    
    TARGET_INDEX = 1
    target_img_fp32, target_label = testset[TARGET_INDEX]
    target_img_fp32 = target_img_fp32.unsqueeze(0) 
    print(f"提取黄金测试图片成功 (Index={TARGET_INDEX})！真实标签: {target_label}")

    print("\n>>> 步骤 2/4: 执行严格的 PTQ 静态校准 (基于训练集)...")
    global_max_in = 0.0
    global_max_out1, global_max_out2, global_max_out3, global_max_out4 = 0.0, 0.0, 0.0, 0.0
    global_max_fc = 0.0

    with torch.no_grad():
        for images, _ in calib_loader:
            global_max_in = max(global_max_in, images.abs().max().item())
            
            out1 = F.conv2d(images, model.conv1.weight, model.conv1.bias, padding=1)
            global_max_out1 = max(global_max_out1, out1.abs().max().item())
            act1 = F.relu(out1)
            
            out2 = F.conv2d(act1, model.conv2.weight, model.conv2.bias, padding=1)
            global_max_out2 = max(global_max_out2, out2.abs().max().item())
            act2 = F.max_pool2d(F.relu(out2), 2)
            
            out3 = F.conv2d(act2, model.conv3.weight, model.conv3.bias, padding=1)
            global_max_out3 = max(global_max_out3, out3.abs().max().item())
            act3 = F.relu(out3)
            
            out4 = F.conv2d(act3, model.conv4.weight, model.conv4.bias, padding=1)
            global_max_out4 = max(global_max_out4, out4.abs().max().item())
            act4 = F.max_pool2d(F.relu(out4), 2)
            
            flat = act4.view(act4.size(0), -1)
            fc_out = F.linear(flat, model.fc.weight, model.fc.bias)
            global_max_fc = max(global_max_fc, fc_out.abs().max().item())

    # 防除零保护算 Scale
    s_in = global_max_in / 127.0 if global_max_in > 0 else 1.0
    s_out1 = global_max_out1 / 127.0 if global_max_out1 > 0 else 1.0
    s_out2 = global_max_out2 / 127.0 if global_max_out2 > 0 else 1.0
    s_out3 = global_max_out3 / 127.0 if global_max_out3 > 0 else 1.0
    s_out4 = global_max_out4 / 127.0 if global_max_out4 > 0 else 1.0
    s_out_fc = global_max_fc / 127.0 if global_max_fc > 0 else 1.0

    print("\n>>> 步骤 3/4: 固化量化参数与权重...")
    w1_q, s_w1 = sym_quantize(model.conv1.weight)
    b1_q = torch.round(model.conv1.bias / (s_in * s_w1)).to(torch.int32)
    mult1, shift1 = calc_multiplier_shift(s_in, s_w1, s_out1)
    
    w2_q, s_w2 = sym_quantize(model.conv2.weight)
    b2_q = torch.round(model.conv2.bias / (s_out1 * s_w2)).to(torch.int32)
    mult2, shift2 = calc_multiplier_shift(s_out1, s_w2, s_out2)
    
    w3_q, s_w3 = sym_quantize(model.conv3.weight)
    b3_q = torch.round(model.conv3.bias / (s_out2 * s_w3)).to(torch.int32)
    mult3, shift3 = calc_multiplier_shift(s_out2, s_w3, s_out3)
    
    w4_q, s_w4 = sym_quantize(model.conv4.weight)
    b4_q = torch.round(model.conv4.bias / (s_out3 * s_w4)).to(torch.int32)
    mult4, shift4 = calc_multiplier_shift(s_out3, s_w4, s_out4)
    
    wfc_q, s_wfc = sym_quantize(model.fc.weight)
    bfc_q = torch.round(model.fc.bias / (s_out4 * s_wfc)).to(torch.int32)
    mult_fc, shift_fc = calc_multiplier_shift(s_out4, s_wfc, s_out_fc)

    print("\n>>> 步骤 4/4: 为靶向目标 (Index=1) 生成 Bit-Accurate 黄金输出...")
    # 严格按照全 INT8 张量流进行前向推演
    target_img_q = torch.clamp(torch.round(target_img_fp32 / s_in), -127, 127).to(torch.int8)

    mac1 = F.conv2d(target_img_q.float(), w1_q.float(), bias=b1_q.float(), padding=1).to(torch.int32)
    q_conv1_out = torch.clamp((mac1 * mult1) >> shift1, -127, 127).to(torch.int8)
    q_act1_out = F.relu(q_conv1_out.float()).to(torch.int8)
    
    mac2 = F.conv2d(q_act1_out.float(), w2_q.float(), bias=b2_q.float(), padding=1).to(torch.int32)
    q_conv2_out = torch.clamp((mac2 * mult2) >> shift2, -127, 127).to(torch.int8)
    q_pool1_out = F.max_pool2d(F.relu(q_conv2_out.float()), 2).to(torch.int8)
    
    mac3 = F.conv2d(q_pool1_out.float(), w3_q.float(), bias=b3_q.float(), padding=1).to(torch.int32)
    q_conv3_out = torch.clamp((mac3 * mult3) >> shift3, -127, 127).to(torch.int8)
    q_act3_out = F.relu(q_conv3_out.float()).to(torch.int8)
    
    mac4 = F.conv2d(q_act3_out.float(), w4_q.float(), bias=b4_q.float(), padding=1).to(torch.int32)
    q_conv4_out = torch.clamp((mac4 * mult4) >> shift4, -127, 127).to(torch.int8)
    q_pool2_out = F.max_pool2d(F.relu(q_conv4_out.float()), 2).to(torch.int8)

    q_flat_in = my_flatten(q_pool2_out)
    mac_fc = F.linear(q_flat_in.float(), wfc_q.float(), bias=bfc_q.float()).to(torch.int32)
    q_fc_out = torch.clamp((mac_fc * mult_fc) >> shift_fc, -127, 127).to(torch.int8)

    predicted_label = torch.argmax(q_fc_out, dim=1).item()
    print(f"🎯 INT8 黄金模型对 Target 的预测结果: {predicted_label} (对比真实标签: {target_label})")

    # ==========================================
    # 5. 导出到大头文件
    # ==========================================
    with open("test_vector.h", "w") as f:
        f.write("#ifndef TEST_VECTOR_H\n#define TEST_VECTOR_H\n\n#include <stdint.h>\n\n")
        f.write("// ================= 全局统一量化参数 =================\n")
        f.write(f"const float GLOBAL_S_IN = {s_in};\n\n")
        
        f.write(f"const int32_t L1_MULT = {mult1};\nconst int32_t L1_SHIFT = {shift1};\n\n")
        f.write(f"const int32_t L2_MULT = {mult2};\nconst int32_t L2_SHIFT = {shift2};\n\n")
        f.write(f"const int32_t L3_MULT = {mult3};\nconst int32_t L3_SHIFT = {shift3};\n\n")
        f.write(f"const int32_t L4_MULT = {mult4};\nconst int32_t L4_SHIFT = {shift4};\n\n")
        f.write(f"const int32_t FC_MULT = {mult_fc};\nconst int32_t FC_SHIFT = {shift_fc};\n\n")
        
        export_c_array(target_img_q, "test_input_img", f, "int8_t")
        
        f.write(f"// ================= Conv1 阶段 =================\n")
        export_c_array(w1_q, "test_conv1_weight", f, "int8_t")
        export_c_array(b1_q, "test_conv1_bias", f, "int32_t")
        f.write(f"// 第一层 Conv + ReLU 后的输出结果 (大小 64x32x32 = 65536)\n")
        export_c_array(q_act1_out, "expected_act1_output", f, "int8_t")
        
        f.write(f"// ================= Conv2 阶段 =================\n")
        export_c_array(w2_q, "test_conv2_weight", f, "int8_t")
        export_c_array(b2_q, "test_conv2_bias", f, "int32_t")
        f.write(f"// 第二层 Conv + ReLU + Pool1 后的输出结果 (大小 64x16x16 = 16384)\n")
        export_c_array(q_pool1_out, "expected_pool1_output", f, "int8_t")
        
        f.write(f"// ================= Conv3 阶段 =================\n")
        export_c_array(w3_q, "test_conv3_weight", f, "int8_t")
        export_c_array(b3_q, "test_conv3_bias", f, "int32_t")
        f.write(f"// 第三层 Conv + ReLU 后的输出结果 (大小 64x16x16 = 16384)\n")
        export_c_array(q_act3_out, "expected_act3_output", f, "int8_t")
        
        f.write(f"// ================= Conv4 阶段 =================\n")
        export_c_array(w4_q, "test_conv4_weight", f, "int8_t")
        export_c_array(b4_q, "test_conv4_bias", f, "int32_t")
        f.write(f"// 第四层 Conv + ReLU + Pool2 后的输出结果 (大小 64x8x8 = 4096)\n")
        export_c_array(q_pool2_out, "expected_pool2_output", f, "int8_t")
        
        f.write(f"// ================= FC 阶段 =================\n")
        export_c_array(wfc_q, "test_fc_weight", f, "int8_t")
        export_c_array(bfc_q, "test_fc_bias", f, "int32_t")
        f.write(f"// 全连接层最终 10 分类输出\n")
        export_c_array(q_fc_out, "expected_fc_output", f, "int8_t")

        f.write("#endif // TEST_VECTOR_H\n")
    print("静态全局量化的黄金对账单导出完毕！")

if __name__ == "__main__":
    main()

>>> 步骤 1/4: 加载融合版模型与数据集...
Files already downloaded and verified
Files already downloaded and verified
提取黄金测试图片成功 (Index=1)！真实标签: 8

>>> 步骤 2/4: 执行严格的 PTQ 静态校准 (基于训练集)...

>>> 步骤 3/4: 固化量化参数与权重...

>>> 步骤 4/4: 为靶向目标 (Index=1) 生成 Bit-Accurate 黄金输出...
🎯 INT8 黄金模型对 Target 的预测结果: 8 (对比真实标签: 8)
静态全局量化的黄金对账单导出完毕！


将test_vector.h中的test_input_img每3个数重新拼接为一个32bit数,输出到act_layer1_packed.h中

In [ ]:
import numpy as np
import re

def parse_c_header_array(filename):
    """
    从 C 语言头文件中提取一维数组，并尝试猜测其维度
    """
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"[错误] 未找到文件 {filename}")
        return None, None

    match = re.search(r'const\s+(int8_t|uint8_t)\s+([a-zA-Z0-9_]+)\[\d+\]\s*=\s*\{([^}]+)\};', content)
    if not match:
        print("[错误] 未匹配到 C 语言数组声明！")
        return None, None

    array_name = match.group(2)
    data_str = match.group(3)
    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    
    return np.array(numbers, dtype=np.int8), array_name

def pack_activation_chw_to_hwc(img_chw):
    """
    核心编译逻辑：将 CHW 格式的激活值转化为 NPU 专属的 HWC 分组 32-bit 格式
    - img_chw: numpy array, 形状为 (C, H, W)
    - 统一规则: 越低的位对应越低的通道数 (IC0 -> [7:0], IC1 -> [15:8]...)
    """
    C, H, W = img_chw.shape
    print(f"[Compiler] 原始特征图形状: Channel={C}, Height={H}, Width={W}")

    # 1. 输入通道数补齐到 4 的倍数 (软件在尾端补零)
    # 因为 IC0 对应低位，新增的补零通道会在最高位 (比如 IC3 -> [31:24])
    pad_c = (4 - (C % 4)) % 4
    if pad_c > 0:
        img_chw = np.pad(img_chw, ((0, pad_c), (0, 0), (0, 0)), mode='constant', constant_values=0)
        print(f"  |- [Info] 通道数 {C} 不是 4 的倍数，已在尾部补 {pad_c} 个 0，当前通道数: {img_chw.shape[0]}")
    
    C_padded = img_chw.shape[0]

    # 2. 维度转置 CHW -> HWC
    img_hwc = np.transpose(img_chw, (1, 2, 0))

    # 3. 模拟物理线束拼接：按 H, W, IC_Group 顺序打包 32-bit 字
    packed_data = []
    ic_groups = C_padded // 4
    
    for y in range(H):
        for x in range(W):
            for ig in range(ic_groups):
                base_c = ig * 4
                # 按顺序取出 4 个通道，IC0 对应 base_c + 0
                c0 = img_hwc[y, x, base_c + 0] & 0xFF
                c1 = img_hwc[y, x, base_c + 1] & 0xFF
                c2 = img_hwc[y, x, base_c + 2] & 0xFF
                c3 = img_hwc[y, x, base_c + 3] & 0xFF
                
                # 严格遵守：越低的位对应越低的通道数
                packed_word = (c3 << 24) | (c2 << 16) | (c1 << 8) | c0
                packed_data.append(packed_word)

    print(f"[Compiler] 打包完成！共生成 {len(packed_data)} 个 32-bit 字。")
    return packed_data, C_padded

def export_to_c_header(packed_data, filename, array_name="npu_act_packed32"):
    """
    导出为 C 语言头文件，供 PicoRV32 或 Testbench 使用
    """
    out_lines = [
        f"// ===================================================================",
        f"// NPU Activation Data",
        f"// 格式: HWC 交织打包, 每 4 个 8-bit Channel 拼成 1 个 32-bit Word",
        f"// 字节序: [31:24]=IC3, [23:16]=IC2, [15:8]=IC1, [7:0]=IC0",
        f"// ===================================================================",
        f"const uint32_t {array_name}[{len(packed_data)}] = {{"
    ]
    
    for i in range(0, len(packed_data), 8):
        chunk = packed_data[i:i+8]
        chunk_str = ", ".join([f"0x{x:08X}" for x in chunk])
        out_lines.append(f"    {chunk_str},")
        
    out_lines.append("};")

    with open(filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines) + "\n")
    print(f"[Export] 成功导出 C 头文件: {filename}")

def export_to_hex(packed_data, filename):
    """
    导出为纯 Hex 文件，供 Verilog $readmemh 使用
    """
    with open(filename, 'w', encoding='utf-8') as f:
        for word in packed_data:
            f.write(f"{word:08x}\n")
    print(f"[Export] 成功导出 Hex 烧录文件: {filename}")


if __name__ == "__main__":
    # ==========================================
    # 用法示例 1：处理现有的第一层 C 头文件图像
    # ==========================================
    print(">>> 任务 1: 打包第一层图像 (RGB 3x32x32) <<<")
    raw_array, original_name = parse_c_header_array("test_vector.h")
    
    if raw_array is not None:
        # 第一层原始图是 3x32x32
        img_chw = raw_array.reshape(3, 32, 32)
        # 调用打包引擎
        packed_words, final_c = pack_activation_chw_to_hwc(img_chw)
        # 导出结果
        export_to_c_header(packed_words, "act_layer1_packed.h", array_name=f"{original_name}_packed32")
        export_to_hex(packed_words, "act_layer1.hex")

>>> 任务 1: 打包第一层图像 (RGB 3x32x32) <<<
[Compiler] 原始特征图形状: Channel=3, Height=32, Width=32
  |- [Info] 通道数 3 不是 4 的倍数，已在尾部补 1 个 0，当前通道数: 4
[Compiler] 打包完成！共生成 1024 个 32-bit 字。
[Export] 成功导出 C 头文件: act_layer1_packed.h
[Export] 成功导出 Hex 烧录文件: act_layer1.hex


## 将权重数据打包成便于给SRAM赋值的格式

In [ ]:
import numpy as np
import re

def parse_c_header_array(filename, target_var):
    """从 C 头文件中提取数组数据"""
    try:
        with open(filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"[错误] 未找到文件 {filename}")
        return None, None
    pattern = rf'const\s+int8_t\s+({target_var})\[\d+\]\s*=\s*\{{([^}}]+)\}};'
    match = re.search(pattern, content)
    # match = re.search(r'const\s+int8_t\s+(test_conv1_weight)\[\d+\]\s*=\s*\{([^}]+)\};', content)
    if not match:
        return None, None

    array_name = match.group(1)
    data_str = match.group(2)
    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    return np.array(numbers, dtype=np.int8), array_name

def pack_weights_for_npu(weight_oihw, sys_cols=4):
    """
    核心逻辑：将 [OC, IC, KH, KW] 的权重重排为 NPU 专用格式
    - sys_cols: 硬件阵列的物理列数 (即最大支持的并行 OC 数量)
    """
    OC, IC, KH, KW = weight_oihw.shape
    print(f"[Compiler] 原始权重形状: OC={OC}, IC={IC}, KH={KH}, KW={KW} | 阵列宽度={sys_cols}")

    # 1. IC 补齐到 4 的倍数
    pad_ic = (4 - (IC % 4)) % 4
    if pad_ic > 0:
        weight_oihw = np.pad(weight_oihw, ((0, 0), (0, pad_ic), (0, 0), (0, 0)), mode='constant')
        print(f"  |- [Info] 输入通道补零: {IC} -> {weight_oihw.shape[1]}")
    
    IC_padded = weight_oihw.shape[1]
    ic_groups = IC_padded // 4

    packed_data = []
    
    # 2. 最外层：OC 分块 (Chunking)
    # 比如 OC=16, sys_cols=4, 则分为 4 个 Chunk: [0~3], [4~7], [8~11], [12~15]
    for oc_start in range(0, OC, sys_cols):
        oc_end = min(oc_start + sys_cols, OC)
        actual_oc_len = oc_end - oc_start
        print(f"  |- [Info] 打包 OC Chunk: {oc_start} ~ {oc_end - 1} (长度: {actual_oc_len})")

        # 3. 严格遵循硬件 S_COMPUTE 的计算顺序：
        # IC组 (最慢) -> KY (中) -> KX (最快)
        for ig in range(ic_groups):
            for ky in range(KH):
                for kx in range(KW):
                    
                    # 4. 最内层：连续输出当前 Chunk 内的每一个 OC
                    # 这样硬件 AXI 读进来时，刚好顺次填入 sa_weight_buffer[0], [1]...
                    for oc in range(oc_start, oc_end):
                        # 取出 4 个 IC (遵守越低位对应越低的通道数)
                        c0 = weight_oihw[oc, ig*4 + 0, ky, kx] & 0xFF
                        c1 = weight_oihw[oc, ig*4 + 1, ky, kx] & 0xFF
                        c2 = weight_oihw[oc, ig*4 + 2, ky, kx] & 0xFF
                        c3 = weight_oihw[oc, ig*4 + 3, ky, kx] & 0xFF
                        
                        packed_word = (c3 << 24) | (c2 << 16) | (c1 << 8) | c0
                        packed_data.append(packed_word)

    print(f"[Compiler] 权重打包完成！共生成 {len(packed_data)} 个 32-bit 字。")
    return packed_data

def pack_bias_for_npu(bias_array, sys_cols=4):
    """
    偏置也必须跟着 OC Chunking 一起分块！
    """
    OC = len(bias_array)
    packed_bias = []
    
    for oc_start in range(0, OC, sys_cols):
        oc_end = min(oc_start + sys_cols, OC)
        
        # 依次写入该 Chunk 的 Bias
        for oc in range(oc_start, oc_end):
            # 将 32-bit 有符号数安全转为 uint32
            val = int(bias_array[oc]) & 0xFFFFFFFF
            packed_bias.append(val)
            
    return packed_bias

def export_to_hex(packed_data, filename, region_name):
    """导出为纯 Hex 文件"""
    with open(filename, 'w', encoding='utf-8') as f:
        for word in packed_data:
            f.write(f"{word:08x}\n")
    print(f"[Export] {region_name} Hex 烧录文件已生成: {filename}")


if __name__ == "__main__":
    # ==========================================
    # 假设你的目标阵列宽度为 4 (如果是 32x32 阵列，改为 32)
    # ==========================================
    TARGET_SYS_COLS = 4

    print(">>> 任务 1: 解析原始 16x3x3x3 权重 <<<")
    # 这里用你原来 test_vector.h 里的数组测试
    raw_array, _ = parse_c_header_array("test_vector.h",target_var="test_conv1_weight")
    
    if raw_array is not None:
        # 重塑为 PyTorch 标准格式: [OC=16, IC=3, KH=3, KW=3]
        weight_oihw = raw_array.reshape(16, 3, 3, 3)
        
        # 调用打包引擎
        packed_weights = pack_weights_for_npu(weight_oihw, sys_cols=TARGET_SYS_COLS)
        export_to_hex(packed_weights, "weight.hex", "Weight")

>>> 任务 1: 解析原始 16x3x3x3 权重 <<<
[Compiler] 原始权重形状: OC=16, IC=3, KH=3, KW=3 | 阵列宽度=4
  |- [Info] 输入通道补零: 3 -> 4
  |- [Info] 打包 OC Chunk: 0 ~ 3 (长度: 4)
  |- [Info] 打包 OC Chunk: 4 ~ 7 (长度: 4)
  |- [Info] 打包 OC Chunk: 8 ~ 11 (长度: 4)
  |- [Info] 打包 OC Chunk: 12 ~ 15 (长度: 4)
[Compiler] 权重打包完成！共生成 144 个 32-bit 字。
[Export] Weight Hex 烧录文件已生成: weight.hex


转化偏置Bias数据为易于给SRAM赋值的格式

In [8]:
import re

def extract_specific_bias(input_filename, output_filename, target_var="test_conv1_bias"):
    # ==========================================
    # 1. 读取原始数据文件
    # ==========================================
    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"错误：未找到文件 {input_filename}，请检查路径。")
        return

    # ==========================================
    # 2. 精确正则匹配 (精准狙击特定的变量名)
    # ==========================================
    # 使用 f-string 动态构建正则表达式，将 target_var 锁定在语法中
    # 注意：这里的 \{{ 和 \}} 是为了在 f-string 中转义大括号
    pattern = rf'const\s+int32_t\s+({target_var})\[\d+\]\s*=\s*\{{([^}}]+)\}};'
    
    match = re.search(pattern, content)
    if not match:
        print(f"❌ 解析失败：未在文件中找到名为 '{target_var}' 的 int32_t 数组！")
        print("请检查变量名拼写是否正确，或数据类型是否为 int32_t。")
        return

    extracted_name = match.group(1)
    data_str = match.group(2)

    # 提取所有带有可选负号的整数
    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    
    if len(numbers) != 16:
        print(f"⚠️ 警告：提取到的数据数量为 {len(numbers)}，预期为 16！")
        return

    # ==========================================
    # 3. 核心转换逻辑 (32-bit 补码及格式化)
    # ==========================================
    out_lines = []
    # 写入 Verilog readmemh 能够识别的地址标记
    out_lines.append("@00000000    // Bias 区域 (Word Index = 0x0 / 4 = 0)")

    for val in numbers:
        # Python 中的整数是无限精度的，负数前面有无限个1。
        # 使用 & 0xFFFFFFFF 将其强制截断为标准的 32-bit 无符号十六进制补码形态。
        hex_val = val & 0xFFFFFFFF
        
        # 格式化输出：08X 保证输出为 8 位大写十六进制，后跟原数值的注释
        out_lines.append(f"{hex_val:08X}     // {val}")

    # ==========================================
    # 4. 写入目标文件
    # ==========================================
    with open(output_filename, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines) + "\n")
        
    print(f"✅ 成功从数组 [{extracted_name}] 精准提取 Bias 数据！")
    print(f"📦 共生成 {len(numbers)} 行 32-bit 格式化数据。")
    print(f"💾 数据已成功保存至：{output_filename}")

if __name__ == "__main__":
    # 执行提取（你可以随时将 "test_conv1_bias" 替换为 "test_conv2_bias"）
    # 假设你的所有常量数组都存在 test_vectors.h 中
    extract_specific_bias("test_vector.h", "bias.hex", target_var="test_conv1_bias")

✅ 成功从数组 [test_conv1_bias] 精准提取 Bias 数据！
📦 共生成 16 行 32-bit 格式化数据。
💾 数据已成功保存至：bias.hex


##将data.hex转化为能够给picorv32+npu联合仿真的头文件test_vector_2.h

In [1]:
import os

# ==========================================
# 1. 配置路径与参数
# ==========================================
INPUT_HEX_FILE = "./sram_data/data.hex"        # 你原来的 Verilog hex 文件
OUTPUT_HEADER_FILE = "test_vector_3.h" # 要生成的 C 语言头文件

# ==========================================
# 2. 核心解析与转换函数
# ==========================================
def convert_hex_to_header(in_file, out_file):
    bias_data = []
    weight_data = []
    image_data = []
    
    current_word_addr = 0
    
    # 2.1 解析 hex 文件
    if not os.path.exists(in_file):
        print(f"[Error] 找不到输入文件: {in_file}")
        return
        
    with open(in_file, "r") as f:
        for line in f:
            # 去除注释 (//) 和两端的空格/换行
            line = line.split("//")[0].strip()
            if not line:
                continue
            
            # 判断是否是地址跳转符 @
            if line.startswith("@"):
                current_word_addr = int(line[1:], 16)
            else:
                # 给数据加上 C 语言的十六进制前缀 0x
                c_hex_val = f"0x{line.lower()}"
                
                # 根据地址（Word Index）归类数据
                if current_word_addr < 0x400:          # @00000000 开始的是 Bias
                    bias_data.append(c_hex_val)
                elif current_word_addr < 0x4000:       # @00000400 开始的是 Weight
                    weight_data.append(c_hex_val)
                else:                                  # @00004000 开始的是 Image
                    image_data.append(c_hex_val)
                
                # 地址自增 1 (Word)
                current_word_addr += 1

    # 2.2 写入 C 头文件
    with open(out_file, "w") as f:
        f.write("// =========================================================================\n")
        f.write("// Auto-generated Header File for PicoRV32 + NPU SoC\n")
        f.write("// =========================================================================\n")
        f.write("#ifndef TEST_VECTOR_H\n")
        f.write("#define TEST_VECTOR_H\n\n")
        f.write("#include <stdint.h>\n\n")
        
        # 辅助写入函数，控制每行排版
        def write_c_array(array_name, data_list):
            # 为了防止 C 编译器报符号溢出警告，用 uint32_t 存储 16 进制常量是最安全的
            f.write(f"const uint32_t {array_name}[{len(data_list)}] = {{\n")
            for i in range(0, len(data_list), 8):
                # 每次取 8 个数据为一行
                chunk = data_list[i:i+8]
                f.write("    " + ", ".join(chunk) + ",\n")
            f.write("};\n\n")
            print(f"[Success] {array_name} 生成完毕，共 {len(data_list)} 个 32-bit Words.")

        write_c_array("test_conv1_bias", bias_data)
        write_c_array("test_conv1_weight_packed32", weight_data)
        write_c_array("test_input_img_packed32", image_data)
        
        # 预留一些我们在 C 代码中需要用到的宏参数
        f.write("// --- Quantization Parameters (示例, 请替换为真实值) ---\n")
        f.write("#define L1_MULT  104\n")
        f.write("#define L1_SHIFT 16\n\n")
        
        f.write("#endif // TEST_VECTOR_H\n")

    print(f"\n[Done] 完美打包！请查看生成的文件: {out_file}")

# 执行脚本
if __name__ == "__main__":
    convert_hex_to_header(INPUT_HEX_FILE, OUTPUT_HEADER_FILE)

[Success] test_conv1_bias 生成完毕，共 16 个 32-bit Words.
[Success] test_conv1_weight_packed32 生成完毕，共 144 个 32-bit Words.
[Success] test_input_img_packed32 生成完毕，共 1024 个 32-bit Words.

[Done] 完美打包！请查看生成的文件: test_vector_3.h
